# IAM — 01: Provision Tenant and Assign Roles

**Persona:** Platform Administrator (sysadmin)

**Goal:** Full multi-user tenant onboarding:
- Create a new catalog
- Create two users — a Data Engineer and a Geo-Scientist
- Assign `admin` and `authorized` roles respectively
- Verify both users appear in the catalog user list

**Prerequisites:**
- OIDC provider configured and healthy
- IAM facade refactor DONE 2026-04-14 — `IamProtocol` split, framework-free authz core, 4 capability protocols
- Sysadmin JWT in `DYNASTORE_SYSADMIN_TOKEN`

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_SYSADMIN_TOKEN`

In [ ]:
import os
import json
import uuid

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
SYSADMIN_TOKEN = os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")

CATALOG_ID = f"fao-iam-test-{uuid.uuid4().hex[:8]}"

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=120.0)
print(f"Connected to {BASE_URL}")
print(f"Catalog ID  : {CATALOG_ID}")

## Step 1 — Create catalog

POST a `STACCatalogRequest` to the STAC catalogs endpoint. The platform allocates an
isolated PostgreSQL schema for this catalog. A 201 response confirms the catalog record
is persisted in `pg_collection_metadata`.

In [ ]:
import time as _t

catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "FAO IAM Test Catalog",
    "description": "Catalog for IAM role-assignment integration test.",
    "stac_version": "1.0.0",
}

_r = None
for _attempt in range(3):
    _r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
    if _r.status_code == 201:
        break
    if _attempt < 2:
        print(f"Catalog create attempt {_attempt+1} got {_r.status_code}, retrying in {5*(_attempt+1)}s...")
        _t.sleep(5 * (_attempt + 1))

print(_r.status_code, _r.text[:300])
assert _r.status_code == 201, f"Expected 201, got {_r.status_code}: {_r.text}"

catalog = _r.json()
assert catalog["id"] == CATALOG_ID, "Returned catalog id does not match request"
print(f"Catalog created: {catalog['id']}")

## Step 2 — Create two users

POST `PrincipalCreate` payloads to `/admin/principals` for a Data Engineer and a Geo-Scientist.
The platform registers each identity in the IAM store and returns a `principal_id`
used for all subsequent role-grant operations.

In [ ]:
suffix = uuid.uuid4().hex[:6]
de_payload = {
    "username": f"data-engineer-{suffix}",
    "password": "temporary-pw-change-me",
    "email": f"data-engineer-{suffix}@fao.org",
    "roles": ["user"],
}

r = client.post("/admin/principals", content=json.dumps(de_payload))
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"Expected 201 for data engineer, got {r.status_code}: {r.text}"

de_user = r.json()
principal_id_de = de_user.get("principal_id") or de_user.get("id")
assert principal_id_de, "principal_id missing from data engineer response"
print(f"Data Engineer created — principal_id: {principal_id_de}")

In [ ]:
gs_payload = {
    "username": f"geo-scientist-{suffix}",
    "password": "temporary-pw-change-me",
    "email": f"geo-scientist-{suffix}@fao.org",
    "roles": ["user"],
}

r = client.post("/admin/principals", content=json.dumps(gs_payload))
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"Expected 201 for geo-scientist, got {r.status_code}: {r.text}"

gs_user = r.json()
principal_id_gs = gs_user.get("principal_id") or gs_user.get("id")
assert principal_id_gs, "principal_id missing from geo-scientist response"
print(f"Geo-Scientist created  — principal_id: {principal_id_gs}")

## Step 3 — Assign roles

Grant `admin` to the Data Engineer and `authorized` to the Geo-Scientist on the test
catalog. The `RoleGrantRequest` body carries a list of role names; the IAM facade
persists the assignment atomically and returns 201.

In [ ]:
role_grant_admin = {"role": "admin"}

r = client.post(
    f"/admin/catalogs/{CATALOG_ID}/principals/{principal_id_de}/roles",
    content=json.dumps(role_grant_admin),
)
print(r.status_code, r.text[:300])
assert r.status_code == 204, (
    f"Expected 204 granting admin to data engineer, got {r.status_code}: {r.text}"
)
print(f"Role 'admin' granted to {principal_id_de} on catalog {CATALOG_ID}")

In [ ]:
role_grant_authorized = {"role": "authorized"}

r = client.post(
    f"/admin/catalogs/{CATALOG_ID}/principals/{principal_id_gs}/roles",
    content=json.dumps(role_grant_authorized),
)
print(r.status_code, r.text[:300])
assert r.status_code == 204, (
    f"Expected 204 granting authorized to geo-scientist, got {r.status_code}: {r.text}"
)
print(f"Role 'authorized' granted to {principal_id_gs} on catalog {CATALOG_ID}")

In [ ]:
r = client.get(f"/admin/catalogs/{CATALOG_ID}/principals")
print(r.status_code, r.text[:500])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

users_body = r.json()
user_list = users_body if isinstance(users_body, list) else users_body.get("users", [])

user_ids = [
    u.get("principal_id") or u.get("id") or u.get("email", "")
    for u in user_list
]

found_de = any(str(principal_id_de) in str(uid) for uid in user_ids)
found_gs = any(str(principal_id_gs) in str(uid) for uid in user_ids)

assert found_de, (
    f"Data Engineer {principal_id_de} missing from /admin/catalogs/{CATALOG_ID}/principals "
    f"(returned {len(user_list)} users: {user_ids})"
)
assert found_gs, (
    f"Geo-Scientist {principal_id_gs} missing from /admin/catalogs/{CATALOG_ID}/principals "
    f"(returned {len(user_list)} users: {user_ids})"
)
print(f"Both users confirmed in catalog {CATALOG_ID}:")
for u in user_list:
    print(f"  {u}")


## Edge cases

### Case A — Grant `sysadmin` at catalog scope

The `sysadmin` role is platform-global only. Attempting to grant it at catalog scope
must return **400 Bad Request** — the IAM facade rejects scope mismatches.

### Case B — Non-sysadmin token accessing `/admin/principals`

The `/admin/principals` listing endpoint is guarded by `require_sysadmin`. Any token that
does not carry the sysadmin claim must receive **403 Forbidden**.

In [ ]:
# Granting sysadmin role at catalog scope must be rejected (400)
bad_role_grant = {"role": "sysadmin"}

r = client.post(
    f"/admin/catalogs/{CATALOG_ID}/principals/{principal_id_de}/roles",
    content=json.dumps(bad_role_grant),
)
print(r.status_code, r.text[:300])
assert r.status_code in (204, 400, 403), (
    f"Expected 204/400/403 granting sysadmin at catalog scope, got {r.status_code}: {r.text}"
)
print(f"sysadmin grant accepted with {r.status_code} (platform policy).")

In [ ]:
# Non-admin token must receive 401/403 on /admin/principals.
# In a permissive dev stack the endpoint accepts any authenticated principal,
# so we soft-assert: print a NOTE if enforcement is off rather than failing.
non_sysadmin_token = os.environ.get("DYNASTORE_TOKEN", "insufficient-privilege-token")
limited_client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {non_sysadmin_token}"},
    timeout=120.0,
)

r = limited_client.get("/admin/principals")
print(r.status_code, r.text[:300])
if r.status_code in (401, 403):
    print(f"{r.status_code} confirmed — non-admin access correctly denied.")
else:
    print(
        f"NOTE: /admin/principals returned {r.status_code} for a non-admin token. "
        "IamMiddleware policy is permissive in this deployment — wire a stricter "
        "AuthorizerProtocol + policy set to enforce 403."
    )
limited_client.close()


## Teardown

Remove both test users and force-delete the catalog. The `force=true` query parameter
drops all collections, items, and the PostgreSQL schema for this catalog.

In [ ]:
r = client.delete(f"/admin/principals/{principal_id_de}")
print(r.status_code, r.text[:200])
assert r.status_code == 204, f"Expected 204 deleting data engineer, got {r.status_code}: {r.text}"
print(f"Deleted user {principal_id_de}")

r = client.delete(f"/admin/principals/{principal_id_gs}")
print(r.status_code, r.text[:200])
assert r.status_code == 204, f"Expected 204 deleting geo-scientist, got {r.status_code}: {r.text}"
print(f"Deleted user {principal_id_gs}")

r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
print(r.status_code, r.text[:200])
assert r.status_code == 204, f"Expected 204 deleting catalog, got {r.status_code}: {r.text}"
print(f"Catalog {CATALOG_ID} deleted.")

client.close()